# ViT From Scratch — Google Colab (GPU) Version

This notebook runs the full ViT-from-scratch project on a free Colab T4 GPU.
Training that takes **2–3 hours on CPU** takes **~10 minutes here**.

**Before running**: Go to `Runtime → Change runtime type → T4 GPU`

---
### What this notebook does
1. Clones / uploads the project code
2. Installs pinned dependencies
3. Trains both the ViT and CNN baseline on CIFAR-10
4. Generates training curves, attention visualizations, and the comparison report
5. Downloads all results as a ZIP

In [ ]:
# ── Step 1: Verify GPU is available ──────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Step 2: Upload project files ─────────────────────────────────────────────
# Option A: Clone from GitHub (if you pushed the project)
# !git clone https://github.com/YOUR_USERNAME/vit-from-scratch.git
# %cd vit-from-scratch

# Option B: Upload a ZIP of the project folder
from google.colab import files
import zipfile, os

print('Upload the vit-from-scratch project as a ZIP file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# Find the extracted directory
dirs = [d for d in os.listdir('.') if os.path.isdir(d) and 'vit' in d.lower()]
if dirs:
    os.chdir(dirs[0])
    print(f'Changed to: {os.getcwd()}')
else:
    print('Could not auto-detect project folder. Set it manually below.')
    # os.chdir('vit-from-scratch')

In [ ]:
# ── Step 3: Install pinned dependencies ──────────────────────────────────────
# Colab already has torch/torchvision but we pin versions for reproducibility.
# If Colab's built-in versions match, this is a no-op.
!pip install -q \
    'torch==2.3.1' \
    'torchvision==0.18.1' \
    'numpy==1.26.4' \
    'matplotlib==3.8.4' \
    'tqdm==4.66.4'

# Verify
import torch, torchvision, matplotlib
print(f'torch={torch.__version__}  torchvision={torchvision.__version__}  matplotlib={matplotlib.__version__}')
print('CUDA:', torch.cuda.is_available())

In [ ]:
# ── Step 4: Run unit tests ───────────────────────────────────────────────────
!python -m pytest tests/ -v --tb=short

In [ ]:
# ── Step 5: Train both models ────────────────────────────────────────────────
# 30 epochs, batch size 128 (larger batch is fine on GPU), lr 3e-4
# Expected time on T4 GPU: ~10-15 minutes total

!python -m src.train --epochs 30 --batch_size 128 --lr 3e-4 --device cuda --num_workers 2

In [ ]:
# ── Step 6: Display training curves ──────────────────────────────────────────
from IPython.display import Image
Image('results/training_curves.png')

In [ ]:
# ── Step 7: Generate attention visualizations ─────────────────────────────────
!python -m src.visualize_attention --num_samples 10 --device cuda

In [ ]:
# Display a few attention visualizations
import os
from IPython.display import Image, display

attn_dir = 'results/attention_visualizations'
imgs = sorted([f for f in os.listdir(attn_dir) if f.endswith('.png') and 'sample_' in f])[:4]
for img_file in imgs:
    print(img_file)
    display(Image(os.path.join(attn_dir, img_file)))

In [ ]:
# ── Step 8: Generate comparison report ───────────────────────────────────────
!python -m src.evaluate --device cuda

In [ ]:
# Print the comparison report inline
with open('results/comparison_report.md') as f:
    print(f.read())

In [ ]:
# ── Step 9: Download all results ─────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('vit_results', 'zip', 'results')
files.download('vit_results.zip')
print('Downloaded results ZIP.')